# Chapter 5 — Evaluating LLM Applications

Hands-On LangChain. Code targets **LangChain 1.x**.

How do you know whether an LLM application is any good — and whether a change (a different model, a
new retrieval strategy) made it better or worse? This chapter builds a small question-answering system
and then evaluates it three ways: by hand, with an LLM as the judge, and by inspecting the chain's
internals.

Companion notebook for Chapter 5. Requires a running Ollama service and `MODEL` in `.env`, and the document at
`../data/career-in-ai.md`. (We build a minimal retrieval QA chain here; Part II covers RAG in depth.)

## 5.0 Build a QA system to evaluate

In [ ]:
from model_provider import get_chat_model, get_embeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter



In [ ]:
# Load the document, split it into chunks, embed them into an in-memory vector store
text = open("../data/career-in-ai.md").read()
docs = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=80).create_documents([text])

embeddings = get_embeddings()
vectorstore = InMemoryVectorStore.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print(f"{len(docs)} chunks embedded")

Now a retrieval-QA chain in plain LCEL: fetch relevant chunks, stuff them into the prompt, answer.

In [ ]:
llm = get_chat_model(temperature=0.0)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

qa_prompt = ChatPromptTemplate.from_template(
    "Use the following context to answer the question. "
    "If you don't know, say you don't know.\n\n"
    "Context:\n{context}\n\nQuestion: {question}"
)

qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | qa_prompt | llm | StrOutputParser()
)
qa_chain.invoke("Who founded DeepLearning.AI?")

## 5.1 Manual evaluation

The simplest approach: hand-write a few questions with known answers (ground truth) and eye-ball the
chain's responses. It is honest and quick, but it does not scale.

In [ ]:
examples = [
    {"query": "Is machine learning foundations the most important skill?", "answer": "Yes"},
    {"query": "What are the Python frameworks you need to learn?", "answer": "TensorFlow and PyTorch"},
]

print(qa_chain.invoke(examples[0]["query"]))

## 5.2 LLM-assisted evaluation

Writing test cases by hand is slow. We can have an LLM **generate** question/answer pairs from the
documents, and then have an LLM **grade** the system's answers. The old `QAGenerateChain` /
`QAEvalChain` helpers are gone in 1.x; we rebuild both with `with_structured_output`.

### Generate question/answer pairs from the documents

In [ ]:
from pydantic import BaseModel, Field

class QAPair(BaseModel):
    """A question and its correct answer, grounded in a document."""
    query: str = Field(description="A question answerable from the document.")
    answer: str = Field(description="The correct answer, grounded in the document.")

gen_prompt = ChatPromptTemplate.from_template(
    "You are a teacher writing a quiz. From the document below, write one question and its "
    "correct answer.\n\nDocument:\n{doc}"
)
qa_generator = gen_prompt | llm.with_structured_output(QAPair)

# generate a QA pair from each of the first few chunks
generated = qa_generator.batch([{"doc": d.page_content} for d in docs[:4]])
for g in generated:
    print("Q:", g.query)
    print("A:", g.answer, "\n")

Add the generated pairs to our hand-written ones to form the full evaluation set.

In [ ]:
examples += [{"query": g.query, "answer": g.answer} for g in generated]
print(f"{len(examples)} examples total")

### Run the QA chain on every example

In [ ]:
predictions = qa_chain.batch([e["query"] for e in examples])
for e, p in zip(examples, predictions):
    print("Q:", e["query"])
    print("Predicted:", p[:120], "...\n")

### Grade the predictions with an LLM judge

We give the judge the question, the ground-truth answer, and the predicted answer, and ask for a
`CORRECT` / `INCORRECT` grade plus a one-line reason.

In [ ]:
from typing import Literal

class Grade(BaseModel):
    """Grade of a predicted answer against the ground truth."""
    grade: Literal["CORRECT", "INCORRECT"] = Field(description="Is the prediction correct?")
    reasoning: str = Field(description="One sentence explaining the grade.")

judge_prompt = ChatPromptTemplate.from_template(
    "You are grading a quiz. Mark the student's answer CORRECT if it is factually consistent with "
    "the true answer, even if worded differently; otherwise INCORRECT.\n\n"
    "Question: {query}\nTrue answer: {answer}\nStudent answer: {prediction}"
)
judge = judge_prompt | llm.with_structured_output(Grade)

graded = judge.batch([
    {"query": e["query"], "answer": e["answer"], "prediction": p}
    for e, p in zip(examples, predictions)
])

In [ ]:
for i, (e, p, g) in enumerate(zip(examples, predictions, graded)):
    print(f"Example {i} — {g.grade}")
    print("  Question :", e["query"])
    print("  Real     :", e["answer"])
    print("  Predicted:", p[:100], "...")
    print("  Why      :", g.reasoning, "\n")

## 5.3 Observing behind the scenes

A final answer alone rarely tells you *why* something went wrong — especially in retrieval, where the
fault is often the fetched context, not the model. `set_debug(True)` makes LangChain print every step:
the inputs, the retrieved context, the exact prompt sent to the model, and the response.

In [ ]:
from langchain_core.globals import set_debug

set_debug(True)
_ = qa_chain.invoke("What does Andrew Ng compare AI to?")
set_debug(False)   # turn it back off

Each model response also carries metadata, including token usage — useful for tracking cost over
time. Calling the model directly (rather than through `StrOutputParser`) exposes it:

In [ ]:
resp = llm.invoke("What does Andrew Ng compare AI to? Answer in one sentence.")
print(resp.content)
print("token usage:", resp.usage_metadata)

> For team-scale evaluation — datasets, automatic scoring, and tracking experiments over time —
> LangChain's hosted companion **LangSmith** provides tracing and evaluators. The manual, generated,
> and LLM-judge techniques in this chapter are the same ideas it automates.